In [ ]:
%%capture
pip install iterative-stratification

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import early_stopping,log_evaluation, Dataset
import lightgbm as lgb
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn import set_config
import warnings
import optuna
from sklearn.preprocessing import PolynomialFeatures,OneHotEncoder
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from scipy.stats import mode
from sklearn.preprocessing import LabelEncoder
from pprint import pprint

warnings.filterwarnings('ignore')

sns.set_theme(style = 'white', palette = 'viridis')
pal = sns.color_palette('viridis')

pd.set_option('display.max_rows', 100)
set_config(transform_output = 'pandas')
pd.options.mode.chained_assignment = None

In [ ]:
data = pd.read_csv('/kaggle/input/playground-series-s4e3/train.csv',low_memory=False)
data.head()

In [ ]:
original = pd.read_csv('/kaggle/input/playgrounds4e03ancillary/PlaygroundS4E3Original.csv',low_memory=False)
original.head()

In [ ]:
data = pd.concat([data,original]).drop('id',axis=1).drop_duplicates()
data.head()

In [ ]:
label_cols = ['Pastry','Z_Scratch','K_Scatch','Stains','Dirtiness','Bumps','Other_Faults']
data = data[data[label_cols].sum(axis=1) <= 1]

In [ ]:
test = pd.read_csv('/kaggle/input/playground-series-s4e3/test.csv',low_memory=False)
test.head()

In [ ]:
data.info()

In [ ]:
class Model:
    def __init__(self, train, test):
        self.train = train
        self.test = test
        self.model_dict = dict()
        self.test_predict_list = list()
        
    def fit(self,params):
        target_col = ['Pastry','Z_Scratch','K_Scatch','Stains','Dirtiness','Bumps','Other_Faults']
        drop_col = ['id']
        
        train_cols = [col for col in self.train.columns.to_list() if col not in target_col + drop_col]
        scores = list()
        
        
        for i in range(4):
            mskf = MultilabelStratifiedKFold(n_splits=20, shuffle=True)
            oof_valid_preds = np.zeros((self.train[train_cols].shape[0], len(target_col)))
                
            for fold, (train_idx, valid_idx) in enumerate(mskf.split(self.train[train_cols], self.train[target_col])):
                X_train, y_train = self.train[train_cols].iloc[train_idx], self.train[target_col].iloc[train_idx]
                X_valid, y_valid = self.train[train_cols].iloc[valid_idx], self.train[target_col].iloc[valid_idx]
                
                model = XGBClassifier(random_state=i+fold,**params)
                    
                model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], 
                          early_stopping_rounds=100,verbose=False)
                    
                valid_preds = model.predict_proba(X_valid)
                oof_valid_preds[valid_idx] = valid_preds
                test_predict = model.predict_proba(self.test[train_cols])
                self.test_predict_list.append(test_predict)
                score = roc_auc_score(y_valid, valid_preds, multi_class="ova")
                self.model_dict[f'fold_{fold}'] = model
                    
            oof_score = roc_auc_score(self.train[target_col], oof_valid_preds, multi_class="ovr")
            print(f"The OOF auc score for iteration {i+1} is {oof_score}")
            scores.append(oof_score)
        return scores,self.test_predict_list
    
    def objective(self,trial):
        target_col = ['Pastry','Z_Scratch','K_Scatch','Stains','Dirtiness','Bumps','Other_Faults']
        drop_col = ['id']
        test_predict_list = list()
        model_dict = dict()
        
        train_cols = [col for col in self.train.columns.to_list() if col not in target_col + drop_col]
        scores = list()
        params = {'grow_policy': 'depthwise',
                  #'num_class':7,
                      'n_estimators': trial.suggest_int('n_estimators', 500, 2000), 
                       'learning_rate': trial.suggest_loguniform('learning_rate', 1e-5, 1), 
                       'gamma': trial.suggest_uniform('gamma', 0.1, 1), 
                       'subsample': trial.suggest_uniform('subsample', 0.5, 1),
                       'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.3, 1), 
                       'max_depth': trial.suggest_int('max_depth', 5, 30), 
                       'min_child_weight': trial.suggest_int('min_child_weight', 1, 25), 
                       'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-8, 10),
                       'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-8, 10),
                       'booster':'gbtree',
                       'verbosity':0,
                       'device_type': 'cuda','tree_method': 'gpu_hist'
                        }
        
        for i in range(1):
            mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True)
            oof_valid_preds = np.zeros((self.train[train_cols].shape[0], len(target_col)))
                
            for fold, (train_idx, valid_idx) in enumerate(mskf.split(self.train[train_cols], self.train[target_col])):
                X_train, y_train = self.train[train_cols].iloc[train_idx], self.train[target_col].iloc[train_idx]
                X_valid, y_valid = self.train[train_cols].iloc[valid_idx], self.train[target_col].iloc[valid_idx]
                
                model = XGBClassifier(random_state=i+fold,**params)
                    
                model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], early_stopping_rounds=100,verbose=False)
                    
                valid_preds = model.predict_proba(X_valid)
                oof_valid_preds[valid_idx] = valid_preds
                test_predict = model.predict_proba(self.test[train_cols])
                test_predict_list.append(test_predict)
                score = roc_auc_score(y_valid, valid_preds, multi_class="ovr")
                model_dict[f'fold_{fold}'] = model
                    
            oof_score = roc_auc_score(self.train[target_col], oof_valid_preds, multi_class="ovr")
            #print(f"The OOF auc score for iteration {i+1} is {oof_score}")
            scores.append(oof_score)
        return np.mean(np.array(scores))
    
    def find_params(self):
        study = optuna.create_study(direction='maximize')
        study.optimize(self.objective,250)

        best_params = study.best_params
        return best_params

In [ ]:
#model = Model(data,test)
#model.find_params()

In [ ]:
params1 = {'grow_policy': 'depthwise',
                  'n_estimators': 1713, 
                  'learning_rate': 0.00676793896727872, 
                  'gamma': 0.4425433619561816, 
                  'subsample': 0.6782713902375049, 
                  'colsample_bytree': 0.38371870139739117, 
                  'max_depth': 5, 
                  'min_child_weight': 4, 
                  'reg_lambda': 1.7864788262454325e-06, 
                  'reg_alpha': 0.5400111178318557,
                  'booster':'gbtree',
                  #'objective':'multi:softmax',
                  'verbosity': 0 ,'device_type': 'cuda','tree_method': 'gpu_hist'
            }
params2 = { 'n_estimators':1800,
            'learning_rate': 0.006,
            'gamma': 0.44,
            'subsample': 0.7,
            'colsample_bytree': 0.38,
            'max_depth': 5,
            'min_child_weight': 4,
            'reg_lambda': 1.8e-06,
            'reg_alpha': 0.54,
            'booster':'gbtree',
           'grow_policy': 'depthwise',
            'verbosity': 0 ,'device_type': 'cuda','tree_method': 'gpu_hist',}
         

params3 = {'n_estimators': 1235,
 'learning_rate': 0.008352405007099802,
 'gamma': 0.6499918347241912,
 'subsample': 0.9116532305497375,
 'colsample_bytree': 0.49334879814671045,
 'max_depth': 7,
 'min_child_weight': 1,
 'reg_lambda': 1.7005084366184795,
 'reg_alpha': 0.0059679946773570774,'device_type': 'cuda','tree_method': 'gpu_hist'
           }

params4 = {'learning_rate': 0.010319075676480546,
           'reg_lambda': 1.1580953972231198e-06,
           'reg_alpha': 5.343085522464864e-07,
           'subsample': 0.8822444206114078,
           'colsample_bytree': 0.29967227888147463,
           'max_depth': 4,
           'n_estimators': 20000,'device_type': 'cuda','tree_method': 'gpu_hist'}


params5 = {
    'grow_policy': 'depthwise',
    'n_estimators': 829,
    'learning_rate': 0.010260565670497695,
    'gamma': 0.16282691057583543,
    'reg_alpha': 0.010492176264956674,
    'reg_lambda': 0.437536781187624,
    'max_depth': 5,
    'min_child_weight': 2,
    'subsample': 0.6971737476610285,
    'colsample_bytree': 0.5115061295805807,'device_type': 'cuda','tree_method': 'gpu_hist'
    
}

params6 = {'learning_rate': 0.015133423389551155,
           'reg_lambda': 5.061311442683435e-07,
           'reg_alpha': 0.08339470088789445,
           'subsample': 0.9071808534081192,
           'colsample_bytree': 0.4721498069968808,
           'max_depth': 5,
           'n_estimators': 15000,
           'tree_method': 'hist',
           'booster': 'gbtree',
           'gamma': 0.0029151004511253622,
           'grow_policy': 'depthwise','device_type': 'cuda','tree_method': 'gpu_hist'}


In [ ]:
model = Model(data,test)
scores1,preds1 = model.fit(params1)
print(f'The average roc-auc score is {np.mean(scores1)}')

model = Model(data,test)
scores2,preds2 = model.fit(params2)
print(f'The average roc-auc score is {np.mean(scores2)}')

model = Model(data,test)
scores3,preds3 = model.fit(params3)
print(f'The average roc-auc score is {np.mean(scores3)}')

model = Model(data,test)
scores4,preds4 = model.fit(params4)
print(f'The average roc-auc score is {np.mean(scores4)}')

model = Model(data,test)
scores5,preds5 = model.fit(params5)
print(f'The average roc-auc score is {np.mean(scores5)}')

model = Model(data,test)
scores6,preds6 = model.fit(params6)
print(f'The average roc-auc score is {np.mean(scores6)}')

In [ ]:
predictions1 = np.mean(preds1,axis=0)
predictions2 = np.mean(preds2,axis=0)
predictions3 = np.mean(preds3,axis=0)
predictions4 = np.mean(preds4,axis=0)
predictions5 = np.mean(preds5,axis=0)
predictions6 = np.mean(preds6,axis=0)

predictions = (predictions1+predictions2+predictions3+predictions4+predictions5+predictions6)/6
submit = pd.DataFrame(predictions, columns=['Pastry','Z_Scratch','K_Scatch','Stains','Dirtiness','Bumps','Other_Faults'])
submit['id'] = test['id']
submit.to_csv('submission.csv',index=False)
submit